# e-means: equations to NumPy

Paper: [Bessa et al., arXiv:2611.09501v1](https://arxiv.org/abs/2611.09501v1). Phase 1 uses tiny CPU arrays for arithmetic checks, not paper-scale empirical reproduction. Install the project with `pip install -e .` before running this notebook. Read `REPRODUCTION_NOTES.md` for paper/reference differences.

In [ ]:
import numpy as np
from e_means import EMeans, EMeansConfig, LloydKMeans, LloydConfig, mse
from e_means.assignment import batch_statistics
from e_means.schedules import cosine_alpha, resolve_recipe
from e_means.state import EMAState, last_epoch_mean


## Counts, sums, and ratio (§3.1 Eqs. 2–4)

> The centroids themselves are never stored

The paper derives centers from EMA sums and counts: $N \leftarrow (1-\alpha)N+\alpha n$, $S \leftarrow (1-\alpha)S+\alpha s$, $c=S/N$. A batch mean EMA without count weighting is a different algorithm.

In [ ]:
c = np.array([[0., 0.], [10., 10.]])
x = np.array([[1., 2.], [3., 4.], [9., 8.]])
labels = np.array([0, 0, 1])
s, n = batch_statistics(x, labels, 2)
N = np.full(2, 2.0)
S = c * N[:, None]
alpha = 0.25
# Inline form of paper Eq. 2; both statistics receive identical weighting.
N_new = (1-alpha)*N + alpha*n
S_new = (1-alpha)*S + alpha*s
state = EMAState.initialize(c, mass=2.0)
state.update(s, n, alpha)
np.testing.assert_allclose(state.centroids(), S_new / N_new[:, None])
np.testing.assert_allclose(N_new, [2., 1.75])


## Cosine scheduling (§3.1 Eq. 5)

For zero-based update $t$, $\alpha(t)=\alpha_0[\epsilon+(1-\epsilon)(1+\cos(\pi t/M))/2]$. The denominator is M, so the last executed value remains above the floor. Configuration permits explicit B/alpha overrides or paper Table 2 laws.

In [ ]:
def literal_alpha(t, M, alpha0, epsilon=0.1):
    return alpha0*(epsilon + (1-epsilon)*(1+np.cos(np.pi*t/M))/2)
values = [cosine_alpha(t, 10, 0.2) for t in range(10)]
np.testing.assert_allclose(values, [literal_alpha(t, 10, 0.2) for t in range(10)])
assert values[0] == 0.2 and values[-1] > 0.02
assert np.all(np.diff(values) < 0)
print(resolve_recipe(EMeansConfig(8, epochs=3), 1000000))


## Revival (Appendix H.1 Eq. 8)

> every five batches

A live donor is sampled with weight proportional to its mass; one child receives alternating factors $1+\delta,1-\delta$, the donor opposite factors. Each gets half the donor mass. This implementation uses the paper's BF16 machine epsilon $\delta=2^{-7}$; the pinned reference uses a different default.

In [ ]:
state = EMAState.initialize(np.array([[2., 4.], [100., 100.]]))
state.counts[:] = [10, 0]
state.sums[:] = state.fallback * state.counts[:, None]
donors, targets = state.revive(np.random.default_rng(0))
assert donors.tolist() == [0] and targets.tolist() == [1]
np.testing.assert_allclose(state.counts, [5, 5])
np.testing.assert_allclose(state.centroids()[0], [2*(1-2**-7), 4*(1+2**-7)])


## Last-epoch means (§3.2)

Accumulate the final pass's moving assignments and compute their exact means. This minimizes SSE for those codes. It does not require a fresh assignment pass, and does not prove improvement against a separate fresh evaluation of pre-finalization EMA positions.

In [ ]:
fallback = np.array([[0., 0.], [10., 10.]])
closing = last_epoch_mean(s, n, fallback)
np.testing.assert_allclose(closing, [[2, 3], [9, 8]])
assert ((x-closing[labels])**2).sum() <= ((x-fallback[labels])**2).sum()


## Full algorithm and baseline (§1, Algorithm 1)

Both solvers use the same initial centers. Here B and alpha are explicit tiny-example choices. Full experiment records and actual results live in `EXPERIMENT_LOG.md`.

In [ ]:
rng = np.random.default_rng(7)
x = np.vstack([rng.normal(-4, 0.3, (40, 4)), rng.normal(4, 0.3, (40, 4))])
init = x[[0, 40]].copy()
lloyd = LloydKMeans(LloydConfig(2)).fit(x, init_centroids=init)
ema = EMeans(EMeansConfig(2, epochs=3, batch_size=8, alpha0=0.2)).fit(x, init_centroids=init)
assert lloyd.cluster_centers_.shape == ema.cluster_centers_.shape == (2, 4)
assert np.isfinite(ema.cluster_centers_).all()
assert np.all(np.diff([r.train_mse for r in lloyd.history_]) <= 1e-10)
np.testing.assert_array_equal(ema.labels_, ema.predict(x))
assert ema.mse_ == mse(x, ema.cluster_centers_)
print({"lloyd_mse": lloyd.mse_, "e_means_mse": ema.mse_})


## Pitfalls and scope

- Eq. 1 averages squared norms per vector, not per coordinate.
- Weight sums and counts by the same alpha.
- Global update count drives both schedule and revival cadence across epochs.
- Last-pass labels differ from final nearest labels; dropped tails have code -1.
- Multiplicative zero-vector splits can coincide.
- Stochastic e-means loss is not guaranteed monotone.
- Tiny synthetic results do not validate the embedding-data scaling laws.
- PyTorch, slow EMA, CUDA optimization and RunPod benchmarks are deferred.


## Phase 2: tensor parity and shadow EMA

Install optional dependencies with `pip install -e ".[torch]"` for this addendum. All checks run on CPU; no GPU result is claimed. The NumPy initialization/order/revival streams are shared, while PyTorch independently performs numeric tensor operations.

For a shadow half-life h in logical updates, the declared convention is $\alpha_s=1-2^{-1/h}$. Appendix I instead parameterizes constant speed as a multiplier of the theoretical cosine floor. Both are explicit options; they are not combined.

In [ ]:
from dataclasses import replace
from e_means.schedules import slow_alpha
half = EMeansConfig(2, finalize="slow_ema", slow_half_life=10)
a_s = slow_alpha(half, 0.2)
assert abs((1-a_s)**10 - 0.5) < 1e-14
factor = replace(half, slow_half_life=None, slow_ema_factor=0.25)
assert abs(slow_alpha(factor, 0.2) - 0.005) < 1e-15


### Independent shadow recurrence (§3.2, Appendix I)

The shadow accumulates assigned-point sums and counts, not main centroid coordinates. Here is the literal constant-step arithmetic. Main assignments remain the source of statistics, and revival reuses main donor/target IDs while splitting each shadow donor's own mass/geometry. The complete authors' slow-EMA routine is unavailable in the audited release; primitive arithmetic comparisons and a scalar oracle support this documented policy.

In [ ]:
N_shadow = np.full(2, 1e-10)
S_shadow = np.array([[0., 0.], [10., 10.]]) * N_shadow[:, None]
batch_n = np.array([2., 1.])
batch_s = np.array([[4., 6.], [9., 8.]])
shadow_alpha = 0.005
N_shadow = (1-shadow_alpha)*N_shadow + shadow_alpha*batch_n
S_shadow = (1-shadow_alpha)*S_shadow + shadow_alpha*batch_s
np.testing.assert_allclose(S_shadow/N_shadow[:,None], [[2,3],[9,8]], atol=1e-6)


In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is not None:
    import torch
    from e_means import TorchEMeans, TorchLloydKMeans
    previous_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        rng = np.random.default_rng(31)
        x_cpu = rng.normal(size=(64, 5))
        init_cpu = x_cpu[:3].copy()
        cfg = EMeansConfig(3, epochs=2, batch_size=8, alpha0=0.2, finalize="slow_ema", slow_ema_factor=0.25)
        numpy_model = EMeans(cfg).fit(x_cpu, init_centroids=init_cpu)
        torch_model = TorchEMeans(cfg, device="cpu").fit(torch.from_numpy(x_cpu), init_centroids=init_cpu)
        np.testing.assert_allclose(torch_model.cluster_centers_.numpy(), numpy_model.cluster_centers_, atol=2e-12, rtol=2e-12)
        np.testing.assert_array_equal(torch_model.last_epoch_labels_.numpy(), numpy_model.last_epoch_labels_)
        plain = EMeans(replace(cfg, finalize="none", slow_ema_factor=None)).fit(x_cpu, init_centroids=init_cpu)
        np.testing.assert_array_equal(numpy_model.ema_centers_, plain.ema_centers_)
        print("CPU tensor parity and shadow assignment isolation passed")
    finally:
        torch.set_num_threads(previous_threads)
else:
    print("Install the torch extra to execute the tensor parity cell")


### Evidence and remaining limits

The Phase 2 log records 120 real-data CPU fits: UCI optical/pen digits, five seeds, two repetitions, six methods. Repeats are averaged within seed before reporting sample SD. Curves and common-FP64 final metrics are saved separately. These datasets do not establish the paper's embedding-scale claims, and CPU RSS includes framework/data overhead. CUDA correctness cases remain guarded until actual hardware is available; no RunPod, Triton or distributed work was performed.